<a href="https://colab.research.google.com/github/SherjinCruz/Traffic-Prediction/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import ast
import numpy as np
import pandas as pd
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load the Datasets
movies = pd.read_csv("/content/drive/MyDrive/DSA/Project/tmdb_5000_movies.csv")
credits = pd.read_csv("/content/drive/MyDrive/DSA/Project/tmdb_5000_credits.csv")

# 2. Merge Datasets on Movie Title
df = movies.merge(credits, on="title")

# 3. Select Relevant Features
# Important fields for content: movie_id, title, overview, genres, keywords, cast, crew
df = df[["movie_id", "title", "overview", "genres", "keywords", "cast", "crew"]]
df.dropna(inplace=True)


# 4. Data Extraction & Parsing (JSON string parsing)
def convert(obj):
    """Extract all 'name' attributes from stringified JSON lists."""
    names = []
    for item in ast.literal_eval(obj):
        names.append(item["name"])
    return names


def convert_cast(obj):
    """Extract top 3 cast members."""
    names = []
    counter = 0
    for item in ast.literal_eval(obj):
        if counter < 3:
            names.append(item["name"])
            counter += 1
        else:
            break
    return names


def fetch_director(obj):
    """Extract director name from crew list."""
    director = []
    for item in ast.literal_eval(obj):
        if item["job"] == "Director":
            director.append(item["name"])
            break
    return director


# Apply extraction helpers
df["genres"] = df["genres"].apply(convert)
df["keywords"] = df["keywords"].apply(convert)
df["cast"] = df["cast"].apply(convert_cast)
df["crew"] = df["crew"].apply(fetch_director)
df["overview"] = df["overview"].apply(lambda x: x.split())


# 5. Collapse Whitespace to Prevent Token Ambiguity
# Example: 'Sam Worthington' -> 'SamWorthington' so 'Sam' doesn't confuse with 'Sam Mendes'
def collapse(L):
    return [i.replace(" ", "") for i in L]


df["genres"] = df["genres"].apply(collapse)
df["keywords"] = df["keywords"].apply(collapse)
df["cast"] = df["cast"].apply(collapse)
df["crew"] = df["crew"].apply(collapse)

# 6. Combine all features into a single 'tags' column
df["tags"] = (
    df["overview"] + df["genres"] + df["keywords"] + df["cast"] + df["crew"]
)

# Retain only required columns
new_df = df[["movie_id", "title", "tags"]].copy()
new_df["tags"] = new_df["tags"].apply(lambda x: " ".join(x).lower())

# 7. Apply Word Stemming (e.g., 'actions', 'action' -> 'action')
ps = PorterStemmer()


def stem(text):
    return " ".join([ps.stem(word) for word in text.split()])


new_df["tags"] = new_df["tags"].apply(stem)

# 8. Vectorization (CountVectorizer / Bag of Words)
cv = CountVectorizer(max_features=5000, stop_words="english")
vector = cv.fit_transform(new_df["tags"]).toarray()

# 9. Compute Cosine Similarity Matrix
similarity = cosine_similarity(vector)


# 10. Recommendation Function
def recommend(movie_title, top_n=5):
    movie_title_lower = movie_title.strip().lower()
    matches = new_df[new_df["title"].str.lower() == movie_title_lower]

    if matches.empty:
        print(f"Movie '{movie_title}' not found in the database.")
        return []

    movie_index = matches.index[0]
    distances = similarity[movie_index]

    # Sort descending by similarity score, skipping index 0 (itself)
    movie_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[
        1 : top_n + 1
    ]

    print(f"\nTop {top_n} recommendations for '{matches.iloc[0]['title']}':")
    recommendations = []
    for rank, (idx, score) in enumerate(movie_list, 1):
        rec_title = new_df.iloc[idx]["title"]
        recommendations.append(rec_title)
        print(f"{rank}. {rec_title} (similarity: {score:.3f})")

    return recommendations


# Example usage:
if __name__ == "__main__":
    recommend("Avatar")
    recommend("The Dark Knight Rises")


Top 5 recommendations for 'Avatar':
1. Aliens vs Predator: Requiem (similarity: 0.287)
2. Aliens (similarity: 0.269)
3. Falcon Rising (similarity: 0.261)
4. Independence Day (similarity: 0.256)
5. Titan A.E. (similarity: 0.250)

Top 5 recommendations for 'The Dark Knight Rises':
1. The Dark Knight (similarity: 0.423)
2. Batman Returns (similarity: 0.325)
3. Batman (similarity: 0.320)
4. Batman Forever (similarity: 0.318)
5. Batman Begins (similarity: 0.318)


In [2]:
import pickle

# Export cleaned movie metadata and similarity matrix
pickle.dump(new_df.to_dict(), open("movies_dict.pkl", "wb"))
pickle.dump(similarity, open("similarity.pkl", "wb"))